# pipeComb_em_v2_001 Colab 실행 노트북

`pipeComb_em_v1_007`의 JSJ9 Word/Char TF-IDF와 OOF stacking 구조를 유지하고, EMV45 tree를 EMV46 tree로 교체한 실험입니다.

- Text view: JSJ9 Word/Char TF-IDF
- Tree view: EMV46
- Model: LinearSVC + XGBoost + LightGBM OOF stacking
- 중복 제거: JSJ9 tree와 별도 EMV45 tree는 사용하지 않음
- 예상 시간: Colab CPU 기준 약 1.5~4.5시간이며 런타임 성능에 따라 더 길어질 수 있음

대회가 제공한 `train.csv`, `test.csv`, `sample_submission.csv`만 업로드합니다. test 데이터는 transform과 최종 예측에만 사용됩니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/wnswlfhvkr-code/AH05dacon6.git'
BRANCH = 'feature/18'
PROJECT_DIR = Path('/content/AH05dacon6')

if not PROJECT_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
os.chdir(PROJECT_DIR)
print('project:', PROJECT_DIR)
print('branch:', BRANCH)

In [ ]:
# 현재 실험에 필요한 최소 패키지만 설치합니다.
packages = [
    'PyYAML>=6.0',
    'numpy>=1.26',
    'pandas>=2.2',
    'scipy>=1.12',
    'scikit-learn>=1.6',
    'xgboost>=3.0',
    'lightgbm>=4.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('dependencies installed')

In [ ]:
# 대회 제공 파일만 업로드합니다. 파일이 이미 있으면 업로드를 생략합니다.
from google.colab import files

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
missing = [name for name in required_files if not (RAW_DIR / name).exists()]

if missing:
    print('다음 파일을 선택하세요:', missing)
    uploaded = files.upload()
    for name, content in uploaded.items():
        destination = RAW_DIR / Path(name).name
        destination.write_bytes(content)

still_missing = [name for name in required_files if not (RAW_DIR / name).exists()]
if still_missing:
    raise FileNotFoundError(f'필수 파일이 없습니다: {still_missing}')
print('data files ready:', required_files)

In [ ]:
# 설정과 등록 상태만 확인합니다. test 데이터는 이 단계에서 읽지 않습니다.
from copy import deepcopy
import yaml

CONFIG_PATH = PROJECT_DIR / 'configs' / 'test_006_pipeComb_em_v2_001.yaml'
with CONFIG_PATH.open(encoding='utf-8') as file:
    config = yaml.safe_load(file)

assert config['preprocessing']['name'] == 'pipeComb_em_v2_001'
assert config['preprocessing'].get('emv46_parameters') is not None
assert config['model']['name'] == 'pipecomb_oof_stacking'
assert config['experiment_design']['tree_view'] == 'EMV46'

# 저장소의 design_only 설정은 보존하고 Colab 실행용 사본만 만듭니다.
runtime_config = deepcopy(config)
runtime_config['execution'].update(status='colab_run', train=True, evaluate=True)
runtime_config['record'].update(
    status='colab_run', training_performed=True, evaluation_performed=True
)
RUNTIME_CONFIG_PATH = Path('/content/test_006_pipeComb_em_v2_001_runtime.yaml')
RUNTIME_CONFIG_PATH.write_text(
    yaml.safe_dump(runtime_config, allow_unicode=True, sort_keys=False),
    encoding='utf-8',
)
print('runtime config:', RUNTIME_CONFIG_PATH)

## 학습 실행

아래 셀부터 실제 5-fold OOF 평가, holdout 평가, 전체 재학습과 submission 생성이 시작됩니다. Colab 세션이 종료되지 않도록 브라우저 연결을 유지하세요.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'src.train', '--config', str(RUNTIME_CONFIG_PATH)],
    cwd=PROJECT_DIR,
    check=True,
)

In [ ]:
# 생성된 submission을 확인하고 내려받습니다.
import pandas as pd

submission_path = (
    PROJECT_DIR / 'data' / 'processed' /
    'test_006_pipeComb_em_v2_001_pipecomb_oof_stacking_submission.csv'
)
if not submission_path.exists():
    raise FileNotFoundError(f'submission이 생성되지 않았습니다: {submission_path}')
display(pd.read_csv(submission_path).head())
files.download(str(submission_path))